# LUCAS grasslands with EUNIS-lvl2 habitat labels
## Data pre-processing of: 

In [2]:
%load_ext autoreload
%autoreload 2

import os
import io
import base64
import re
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, HTML
from matplotlib_venn import venn2

from data_viz import step_plot_distribution, pie_chart_multilabels_proportion, pie_chart_simple, bar_plot_distribution_with_floating_text, bar_plot_habitats_distribution_VS_syntaxons, stack_bars_habitats
from stats import get_basic_stats

In [3]:
def fig_to_img(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight')
    buf.seek(0)
    data = base64.b64encode(buf.read()).decode("utf-8")
    return f'<img src="data:image/png;base64,{data}" style="margin-right:20px;">'

def display_plots_side_by_side(figs):
    html = f"""<div style="display:flex; align-items:flex-start;">"""
    for fig in figs:
        html += f'{fig_to_img(fig)}'
    html += '\n</div>'
    display(HTML(html))

In [4]:
ROOT_PATH_OCCURRENCES = 'data/output/csv/'

df_occurrences_18_raw = pd.read_csv(os.path.join(ROOT_PATH_OCCURRENCES, 'lucas18.csv'), low_memory=False)
df_exif_18_raw = pd.read_csv(os.path.join(ROOT_PATH_OCCURRENCES, 'lucas18_exif.csv'), low_memory=False)
df_occurrences_22_raw = pd.read_csv(os.path.join(ROOT_PATH_OCCURRENCES, 'lucas22.csv'), low_memory=False)
df_exif_22_raw = pd.read_csv(os.path.join(ROOT_PATH_OCCURRENCES, 'lucas22_exif.csv'), low_memory=False)

In [5]:
print(f'\033[1m● \x1B[4mOccurences 2018\x1B[0m\033[0m: \n > \x1B[4mShape\x1B[0m: {df_occurrences_18_raw.shape} \n > \x1B[4mColumns\x1B[0m: {df_occurrences_18_raw.columns.tolist()}')
display(df_occurrences_18_raw.head(1))
print(f'\033[1m● \x1B[4mExif 2018\x1B[0m\033[0m: \n > \x1B[4mShape\x1B[0m: {df_exif_18_raw.shape} \n > \x1B[4mColumns\x1B[0m: {df_exif_18_raw.columns.tolist()}')
display(df_exif_18_raw.head(1))
print(f'\033[1m● \x1B[4mOccurences 2022\x1B[0m\033[0m: \n \x1B[4mShape\x1B[0m: {df_occurrences_22_raw.shape} \n \x1B[4mColumns\x1B[0m: {df_occurrences_22_raw.columns.tolist()}')
display(df_occurrences_22_raw.head(1))
print(f'\033[1m● \x1B[4mExif 2022\x1B[0m\033[0m: \n > \x1B[4mShape\x1B[0m: {df_exif_22_raw.shape} \n > \x1B[4mColumns\x1B[0m: {df_exif_22_raw.columns.tolist()}')
display(df_exif_22_raw.head(1))

● Occurences 2018: 
 > Shape: (2622, 128) 
 > Columns: ['fid', 'point_id', 'survey_grass_gps_lat', 'survey_grass_gps_lon', 'survey_grass_gps_ew', 'gps_status', 'distancetothloc', 'point_grass_region', 'point_grass_region_name', 'point_altitude', 'areacode', 'alt_class', 'nuts0', 'lc1', 'survey_date', 'survey_grass_cando', 'survey_grass_trnsct_shift', 'survey_grass_trnsct_shift_dist', 'survey_grass_trnsct_direction', 'survey_grass_trnsct_length', 'survey_grass_slope', 'survey_grass_eunis_habitat', 'survey_grass_orientation', 'survey_grass_site_moisture', 'survey_grass_surface', 'survey_grass_animal_paths', 'survey_grass_fertiliz', 'survey_grass_fertiliz_type', 'survey_grass_grassland_type', 'survey_grass_meadow_growth', 'survey_grass_pasture_cattle', 'survey_grass_pasture_horses', 'survey_grass_pasture_sheep', 'survey_grass_pasture_goats', 'survey_grass_pasture_donkeys', 'survey_grass_pasture_pigs', 'survey_grass_pasture_geese', 'survey_grass_pasture_deer', 'survey_grass_other_type', 's

,fid,point_id,survey_grass_gps_lat,survey_grass_gps_lon,survey_grass_gps_ew,gps_status,distancetothloc,point_grass_region,point_grass_region_name,point_altitude,...,survey_grass_woody_other_perc,survey_grass_woody_dead_perc,survey_grass_herb_layer2_h_cm,survey_grass_herb_layer3_h_cm,survey_grass_herb_layer4_h_cm,survey_grass_herb_layer5_h_cm,survey_grass_pasture_grazing,survey_grass_trnsct_start,nuts3,geom
0,1,26521776,37.19606,-8.860353,2,Original,3.715658,5,Mediterranean - West + Central,31,...,NaN,NaN,5.0,NaN,NaN,NaN,After 1st grazing,On the point,PT150,0101000020E6100000EE073C3080B821C0E2067C7E1899...


● Exif 2018: 
 > Shape: (21421, 65) 
 > Columns: ['sourcefile', 'filesize', 'filetype', 'make', 'model', 'xresolution', 'yresolution', 'resolutionunit', 'exposuretime', 'fnumber', 'datetimeoriginal', 'createdate', 'focallength', 'focallengthin35mmformat', 'imagewidth', 'imageheight', 'fov', 'focallength35efl', 'digitalzoom', 'cameraorientation', 'aspectratio', 'gpsdatestamp', 'gpstimestamp', 'gpsdatetime', 'gpslatituderef', 'gpslongituderef', 'gpsaltituderef', 'gpsspeedref', 'gpsspeed', 'gpsimgdirectionref', 'gpsimgdirection', 'gpsdestbearingref', 'gpsdestbearing', 'gpshpositioningerror', 'gpsaltitude', 'gpslatitude', 'gpslongitude', 'gpsposition', 'gpsversionid', 'gpssatellites', 'gpsmapdatum', 'gpsprocessingmethod', 'gpsstatus', 'gpsmeasuremode', 'gpsdop', 'gpsareainformation', 'gpsdifferential', 'gpstrackref', 'gpstrack', 'usercomment', 'devicemanufacturer', 'devicemodel', 'deviceattributes', 'imagequality', 'advancedscenetype', 'jpegquality', 'TimeStamp', 'digitalzoomratio', 'point

,sourcefile,filesize,filetype,make,model,xresolution,yresolution,resolutionunit,exposuretime,fnumber,...,jpegquality,TimeStamp,digitalzoomratio,point_id,img_letter_group,survey_type,orientation,gpsdestlatituderef,gpsdestlongituderef,gpsdestdistanceref
0,/eos/jeodpp/data/projects/REFOCUS/data/LUCAS20...,306915,JPEG,NaN,NaN,1.0,1.0,0.0,NaN,NaN,...,NaN,NaN,NaN,49343014,M,surveyor,NaN,NaN,NaN,NaN


● Occurences 2022: 
 Shape: (12119, 116) 
 Columns: ['point_id', 'user_id', 'point_nuts0', 'pi_extension', 'point_ex_ante', 'point_lat', 'point_long', 'point_altitude', 'point_copernicus', 'point_grassland', 'grass_region_name', 'point_grass_subregion', 'point_grassland_extended', 'point_erosion', 'point_lf', 'point_soil', 'point_soil_organic', 'point_soil_bulk_0_10', 'point_soil_bulk_10_20', 'point_soil_bulk_20_30', 'point_soil_bio', 'survey_date', 'survey_start_time', 'survey_end_time', 'survey_car_latitude', 'survey_car_longitude', 'survey_gps_lat', 'survey_gps_long', 'survey_gps_altitude', 'survey_gps_prec', 'survey_gps_proj', 'SURVEY_GPS_PROJ.1', 'survey_obs_dist', 'survey_calc_dist', 'survey_obs_type', 'survey_homplot_fills_extwin', 'survey_obs_direct', 'survey_reason_dirchange', 'survey_reason_border_from', 'survey_reason_border_to', 'survey_reason_lf', 'survey_lc1', 'survey_lc1_spec', 'survey_lc1_perc', 'survey_lc2', 'survey_lc2_spec', 'survey_lc2_perc', 'survey_lu1', 'survey_l

,point_id,user_id,point_nuts0,pi_extension,point_ex_ante,point_lat,point_long,point_altitude,point_copernicus,point_grassland,...,survey_grass_richness_spec12,survey_grass_richness_spec7,survey_grass_struc_spec1_perc,survey_grass_struc_spec3_perc,survey_grass_struc_spec5_perc,survey_grass_struc_spec7_perc,survey_grass_struc_spec8_perc,survey_grass_struc_spec10_perc,survey_grass_legume_total_perc,geom_surv_4326
0,26421766,PTSU012,PT,0,0,37.086044,-8.945345,140,1,1,...,0,0,0 %,>3 - 8 %,0 %,0 %,0 %,0 %,0 %,0101000020E61000000000006004E421C000000080038B...


● Exif 2022: 
 > Shape: (49699, 63) 
 > Columns: ['sourcefile', 'filesize', 'filetype', 'resolutionunit', 'xresolution', 'yresolution', 'imagedescription', 'make', 'model', 'orientation', 'exposuretime', 'datetimeoriginal', 'createdate', 'aperturevalue', 'focallength', 'gpslatituderef', 'gpslongituderef', 'gpsaltituderef', 'gpstimestamp', 'gpsdatestamp', 'Country-PrimaryLocationCode', 'Country-PrimaryLocationName', 'headline', 'credit', 'Source', 'imagewidth', 'imageheight', 'aperture', 'gpsaltitude', 'gpsdatetime', 'gpslatitude', 'gpslongitude', 'focallength35efl', 'gpsposition', 'fnumber', 'usercomment', 'digitalzoomratio', 'focallengthin35mmformat', 'gpsversionid', 'fov', 'devicemanufacturer', 'devicemodel', 'deviceattributes', 'gpsspeedref', 'gpsspeed', 'gpsprocessingmethod', 'gpsimgdirectionref', 'gpsimgdirection', 'gpsmapdatum', 'gpsdestbearingref', 'gpsdestbearing', 'gpshpositioningerror', 'warning', 'gpssatellites', 'gpsstatus', 'gpsmeasuremode', 'gpstrackref', 'gpsdestlatitude

,sourcefile,filesize,filetype,resolutionunit,xresolution,yresolution,imagedescription,make,model,orientation,...,gpssatellites,gpsstatus,gpsmeasuremode,gpstrackref,gpsdestlatituderef,gpsdestlongituderef,gpsdestdistanceref,gpsareainformation,aspectratio,point_id
0,/eos/jeodpp/data/projects/REFOCUS/data/LUCAS20...,398602,JPEG,0,1,1,"LUCAS 2022, 42862684, GRASS_Vigour",samsung,SM-G390F,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42862684


## Cleaning CSVs

In [6]:
def clean_file_paths(s, prefix):
    suffix = '/'.join(s.split('/')[-4:])
    return os.path.join(prefix, suffix)

In [8]:
df_exif_18 = df_exif_18_raw.copy()
df_exif_22 = df_exif_22_raw.copy()
df_exif_18['sourcefile'] = df_exif_18_raw['sourcefile'].apply(clean_file_paths, prefix='data/output/data/LUCAS2018_Grasslands/')
df_exif_22['sourcefile'] = df_exif_22_raw['sourcefile'].apply(clean_file_paths, prefix='data/output/data/LUCAS2022_Grasslands/')

print(f"Filepath from LUCAS18 changed from \"{df_exif_18_raw['sourcefile'].iloc[0]}\" to \"{df_exif_18['sourcefile'].iloc[0]}\"")
print(f"Filepath from LUCAS22 changed from \"{df_exif_22_raw['sourcefile'].iloc[0]}\" to \"{df_exif_22['sourcefile'].iloc[0]}\"")

Filepath from LUCAS18 changed from "/eos/jeodpp/data/projects/REFOCUS/data/LUCAS2018_Grassland/images/surveyor_photo//CZ/493/430/49343014M.jpg" to "data/output/data/LUCAS2018_Grasslands/CZ/493/430/49343014M.jpg"
Filepath from LUCAS22 changed from "/eos/jeodpp/data/projects/REFOCUS/data/LUCAS2022_photos/Grassland_photos//grasslandPhotos/AT/428/626/202242862684GRASS_Vigour.jpg" to "data/output/data/LUCAS2022_Grasslands/AT/428/626/202242862684GRASS_Vigour.jpg"


## Verifying files

Let's verify if all the listed photos are properly downloaded.

In [11]:
def verify_image_integrity(fp):
    exists = os.path.exists(fp)
    weight = os.path.getsize(fp)
    empty = weight < 100
    return exists, empty, fp

def verify_images_integrity(df, fp_col):
    verifs = {'valid': [],
              'invalid': {'reason': [],
                          'point_id': [],
                          'fp': []}}
    for rowi, row in tqdm(df.iterrows(), total=len(df)):
        exists, empty, fp = verify_image_integrity(row[fp_col])
        if exists and not empty:  # A ∩ B_bar
            verifs['valid'].append(fp)
        if not exists or empty:  # A_bar ∪ B
            verifs['invalid']['fp'].append(fp)
            verifs['invalid']['point_id'].append(row['point_id'])
            if not exists:
                verifs['invalid']['reason'].apend('File does not exist')
            if empty:
                verifs['invalid']['reason'].append('File is empty (no data)')
    all_imgs_valid =  len(verifs['valid']) == len(df)
    return all_imgs_valid, verifs

In [12]:
print(f'Verifying LUCAS 2018 grasslands habitat dataset...')
all_imgs_valid18, verifs18 = verify_images_integrity(df_exif_18, 'sourcefile')
print(f'Verifying LUCAS 2022 grasslands habitat dataset...')
all_imgs_valid22, verifs22 = verify_images_integrity(df_exif_22, 'sourcefile')

if all_imgs_valid18 and all_imgs_valid22:
    print(f'All images exist downloaded and verified !')
else:
    print("Some images were not valid.\n- 2018:")
    for reason, point_id, fp in zip(verifs18["invalid"]['reason'], verifs18["invalid"]['point_id'], verifs18["invalid"]['fp']):
        print(f'{reason}: {point_id}, {fp}')
    print("- 2022: ")
    for reason, point_id, fp in zip(verifs22["invalid"]['reason'], verifs22["invalid"]['point_id'], verifs22["invalid"]['fp']):
        print(f'{reason}: {point_id}, {fp}')

Verifying LUCAS 2018 grasslands habitat dataset...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21421/21421 [00:00<00:00, 27919.65it/s]


Verifying LUCAS 2022 grasslands habitat dataset...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 49699/49699 [00:01<00:00, 28355.58it/s]

Some images were not valid.
- 2018:
File is empty (no data): 36582596, data/output/data/LUCAS2018_Grasslands/FR/365/825/36582596U.jpg
File is empty (no data): 39962408, data/output/data/LUCAS2018_Grasslands/FR/399/624/39962408M.jpg
File is empty (no data): 44382522, data/output/data/LUCAS2018_Grasslands/IT/443/825/44382522R.jpg
File is empty (no data): 47763500, data/output/data/LUCAS2018_Grasslands/PL/477/635/47763500R.jpg
File is empty (no data): 27841918, data/output/data/LUCAS2018_Grasslands/PT/278/419/27841918Q.jpg
File is empty (no data): 28201984, data/output/data/LUCAS2018_Grasslands/PT/282/019/28201984M.jpg
File is empty (no data): 53162586, data/output/data/LUCAS2018_Grasslands/RO/531/625/53162586Q.jpg
File is empty (no data): 54782658, data/output/data/LUCAS2018_Grasslands/RO/547/826/54782658Q.jpg
File is empty (no data): 56762572, data/output/data/LUCAS2018_Grasslands/RO/567/625/56762572M.jpg
File is empty (no data): 45543610, data/output/data/LUCAS2018_Grasslands/SE/455/43

In [20]:
for verifs, df_exif, year in zip([verifs18, verifs22], [df_exif_18, df_exif_22], ['2018', '2019']):
    n_occur = df_exif.shape[0]
    for fp, point_id in zip(verifs["invalid"]['fp'], verifs["invalid"]['point_id']):
        df_exif = df_exif.drop(df_exif[df_exif['sourcefile'] == fp].index)
    match year:
        case '2018':
            df_exif_18_verif = df_exif.copy()
        case '2022':
            df_exif_22_verif = df_exif.copy()
    delta = df_exif.shape[0] - n_occur
    print(f'LUCAS_{year} nb of occurrences after validity check: {n_occur} ->  {df_exif.shape[0]} ({delta} invalid files removed)')

LUCAS_2018 nb of occurrences after validity check: 21421 ->  21411 (-10 invalid files removed)
LUCAS_2019 nb of occurrences after validity check: 49699 ->  49699 (0 invalid files removed)


In [23]:
uhabitats_18 = df_occurrences_18_raw['survey_grass_eunis_habitat'].nunique()
uhabitats_22 = df_occurrences_22_raw['survey_grass_eunis_habitat_type'].nunique()

print(f'Nb of images in LUCAS18 grasslands: {df_exif_18_verif.shape[0]}, in LUCAS22 grasslands: {df_exif_22.shape[0]}')
print(f'Nb of plots in LUCAS18 grasslands: {df_exif_18_verif['point_id'].nunique()}, in LUCAS22 grasslands: {df_exif_22['point_id'].nunique()}')
print(f'Nb of unique EUNIS-lvl2 habitats in LUCAS18 grasslands: {uhabitats_18}, in LUCAS22 grasslands: {uhabitats_22}')

Nb of images in LUCAS18 grasslands: 21411, in LUCAS22 grasslands: 49699
Nb of plots in LUCAS18 grasslands: 2472, in LUCAS22 grasslands: 9400
Nb of unique EUNIS-lvl2 habitats in LUCAS18 grasslands: 28, in LUCAS22 grasslands: 24
